```text
|''||''| '||    
   ||     ||    
   ||     ||    
   ||     ||    
'..|'    .||...|
```

# Exportar Tabla de SQL Server a CSV con Polars

Este notebook utiliza **Polars**, **SQLAlchemy** y **PyODBC** para extraer datos de SQL Server de forma rápida, previniendo bloqueos de base de datos y caídas indefinidas por problemas de red (con un control de timeout estricto).

## 1. Configuración de Variables

In [9]:
# === CONFIGURA TUS VARIABLES AQUÍ ===
SERVER = "localhost"                  # Nombre o IP del servidor SQL Server
DATABASE = "KddPuebla"          # Nombre de la base de datos
TABLA = "entidad_21"                      # Nombre de la tabla que deseas exportar

WINDOWS_AUTH = False                    # Pon True si usas Autenticación de Windows
USUARIO = "sa"                  # Tu usuario (ignorar si WINDOWS_AUTH=True)
CONTRASEÑA = "admin"            # Tu contraseña (ignorar si WINDOWS_AUTH=True)

DRIVER = "ODBC Driver 17 for SQL Server" # Driver instalado en tu sistema
TIMEOUT_CONEXION = 15                   # Limitar tiempo de espera de conexión a 15 segundos (evita colgarse)
EVITAR_BLOQUEOS = True                  # Agrega 'WITH (NOLOCK)' para no atascarse si la tabla está bloqueada
CARPETA_SALIDA = "csv_exportados"       # Carpeta de destino para los archivos CSV (se creará automáticamente)

## 2. Ejecución del Proceso (Con trazado paso a paso)

In [11]:
import polars as pl
from sqlalchemy import create_engine
import urllib.parse
import os

# 1. Codificar contraseña en formato URI (evita errores con caracteres especiales como @, /, :)
pwd_encoded = urllib.parse.quote_plus(CONTRASEÑA)

# 2. Construir la URI de conexión para SQLAlchemy
if WINDOWS_AUTH:
    conn_uri = f"mssql+pyodbc://@{SERVER}/{DATABASE}?driver={DRIVER}&trusted_connection=yes"
else:
    conn_uri = f"mssql+pyodbc://{USUARIO}:{pwd_encoded}@{SERVER}/{DATABASE}?driver={DRIVER}"

# 3. Configurar el motor de conexión con timeout de inicio de sesión
engine = create_engine(
    conn_uri,
    connect_args={"timeout": TIMEOUT_CONEXION} # Detiene el proceso rápido si no puede conectar
)

# 4. Construir la consulta SQL
if EVITAR_BLOQUEOS:
    # 'WITH (NOLOCK)' evita que la consulta espere si otra transacción está modificando la tabla
    query = f"SELECT * FROM {TABLA} WITH (NOLOCK)"
else:
    query = f"SELECT * FROM {TABLA}"

archivo_csv = f"{TABLA}_exportada.csv"
ruta_archivo = os.path.join(CARPETA_SALIDA, archivo_csv)

print(f"[*] Iniciando proceso de exportación para la tabla '{TABLA}'...")

try:
    # Paso A: Intentar establecer conexión
    print(f"[+] Intentando conectar al servidor '{SERVER}' (Límite: {TIMEOUT_CONEXION} segundos)...")
    with engine.connect() as conn:
        print("[+] ¡Conexión establecida correctamente!")
        
        # Paso B: Ejecutar y transferir los datos
        print(f"[+] Ejecutando consulta: '{query}'")
        print("    (Descargando filas, por favor espera...)")
        df = pl.read_database(query, connection=conn)
    
    # Paso C: Guardar los datos obtenidos en CSV
    print(f"[+] Descarga finalizada: se leyeron {df.height} filas y {df.width} columnas.")
    print(f"[+] Guardando datos en el archivo '{ruta_archivo}'...")
    os.makedirs(CARPETA_SALIDA, exist_ok=True)
    df.write_csv(ruta_archivo)
    print(f"[+] ¡Exportación exitosa! Guardado en: {os.path.abspath(ruta_archivo)}")
    
except Exception as e:
    print("\n[!] ERROR DETECTADO - EL PROCESO FALLÓ:")
    print(f"Detalle: {e}")
    print("\nSugerencias de diagnóstico:")
    print(" 1. Verifica si el nombre del servidor (SERVER) y la base de datos (DATABASE) son correctos.")
    print(" 2. Revisa que el driver ODBC especificado coincida con el instalado en tu sistema.")
    print(" 3. Si la conexión falló rápido por timeout, confirma que el servidor acepte conexiones TCP e IP y que el firewall no esté bloqueando el puerto 1433.")
    print(" 4. Si falló después de conectar, asegúrate de que la tabla '{TABLA}' exista y tengas permisos para leerla.")

[*] Iniciando proceso de exportación para la tabla 'entidad_21'...
[+] Intentando conectar al servidor 'localhost' (Límite: 15 segundos)...
[+] ¡Conexión establecida correctamente!
[+] Ejecutando consulta: 'SELECT * FROM entidad_21 WITH (NOLOCK)'
    (Descargando filas, por favor espera...)
[+] Descarga finalizada: se leyeron 722660 filas y 46 columnas.
[+] Guardando datos en el archivo 'csv_exportados\entidad_21_exportada.csv'...
[+] ¡Exportación exitosa! Guardado en: d:\Users\JoseLuis\Desktop\UTP\Extracción de conocimiento db\Polars\csv_exportados\entidad_21_exportada.csv
